In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# Monotone Convex interpolation — Hagan-West 2006

def mc_node_fwds(t, r):
    # t, r: sorted knot arrays; returns (fd, f)
    # f: discrete forwards on each segment (length n-1)
    # fd: continuous node forwards at each knot (length n)
    n = len(t)
    f = np.diff(r * t) / np.diff(t)
    fd = np.empty(n)
    fd[0] = f[0]
    fd[-1] = f[-1]
    for i in range(1, n - 1):
        fd[i] = (f[i-1]*(t[i+1]-t[i]) + f[i]*(t[i]-t[i-1])) / (t[i+1]-t[i-1])
    return fd, f


def mc_yield_at(x, t, r, fd, f):
    # scalar query x in years; returns interpolated yield
    if x <= t[0]: return float(r[0])
    if x >= t[-1]: return float(r[-1])
    i = min(np.searchsorted(t, x, side='right') - 1, len(t) - 2)
    t0, h, fi = t[i], t[i+1] - t[i], f[i]
    g0, g1 = fd[i] - fi, fd[i+1] - fi
    s = (x - t0) / h
    # area under forward from t0 to x (H-W eq. 4.2)
    area = fi*h*s + g0*h*(s - 2*s**2 + s**3) + g1*h*(-s**2 + s**3)
    return (r[i]*t0 + area) / x


def mc_interpolate(query, knot_t, knot_r):
    t = np.asarray(knot_t, float)
    r = np.asarray(knot_r, float)
    if len(t) == 1:
        return np.full(len(query), r[0])
    fd, f = mc_node_fwds(t, r)
    return np.array([mc_yield_at(x, t, r, fd, f) for x in query])

In [ ]:
CMT_TENORS = list(range(5, 16))
TENOR_COLS = [f'{t}Y' for t in CMT_TENORS]


def flag_otr(sub):
    sub = sub.copy()
    sub['bucket'] = sub['years_to_maturity'].round().astype(int)
    idx = sub.groupby('bucket')['issue_date'].idxmax()
    return sub.loc[idx].sort_values('years_to_maturity').reset_index(drop=True)


def fill_tenors(rec):
    # forward-fill left→right, then back-fill any leading NaNs with first valid
    vals = [rec[c] for c in TENOR_COLS]
    last = None
    for j in range(len(vals)):
        if not np.isnan(vals[j]): last = vals[j]
        elif last is not None: vals[j] = last
    first = next((v for v in vals if not np.isnan(v)), None)
    if first is not None:
        for j in range(len(vals)):
            if np.isnan(vals[j]): vals[j] = first
            else: break
    for c, v in zip(TENOR_COLS, vals):
        rec[c] = v


def build_country_cmt(dates, panel, meta):
    meta_r = meta.reset_index(drop=True)
    records = []
    for obs in dates:
        mask = (meta_r['issue_date'] <= obs) & (meta_r['maturity_date'] > obs)
        if not mask.any():
            continue
        row_vals = panel.loc[[obs]].iloc[0].values
        sub = meta_r[mask].copy().reset_index(drop=True)
        sub['yield'] = row_vals[mask.values]
        sub = sub[sub['yield'].notna()].copy()
        if sub.empty:
            continue
        sub['years_to_maturity'] = (sub['maturity_date'] - obs).dt.days / 365.0
        otr = flag_otr(sub)
        if len(otr) < 2:
            continue
        kt, ky = otr['years_to_maturity'].values, otr['yield'].values
        lo, hi = kt[0], kt[-1]
        rec = {'Fecha': obs}
        for tenor in CMT_TENORS:
            rec[f'{tenor}Y'] = mc_interpolate([float(tenor)], kt, ky)[0] if lo <= tenor <= hi else np.nan
        fill_tenors(rec)
        records.append(rec)
    if not records:
        return pd.DataFrame(columns=['Fecha'] + TENOR_COLS)
    return pd.DataFrame(records)[['Fecha'] + TENOR_COLS].reset_index(drop=True)

In [ ]:
FILEPATH = 'df_rv.xlsx'
raw = pd.read_excel(FILEPATH, sheet_name='LC', header=None)

bond_cols = raw.columns[1:]  # integer column positions in raw for bonds

# meta uses 0-based positional index matching data_raw bond columns below
meta = pd.DataFrame({
    'type':          raw.loc[0, bond_cols].values,
    'issue_date':    pd.to_datetime(raw.loc[1, bond_cols].values, dayfirst=True, errors='coerce'),
    'maturity_date': pd.to_datetime(raw.loc[2, bond_cols].values, dayfirst=True, errors='coerce'),
    'daycount':      raw.loc[3, bond_cols].values.astype(float).astype(int),  # int: 360/365/252
    'ticker':        raw.loc[4, bond_cols].values.astype(str),
})

# yield panel: row per date, columns = 0..n_bonds-1 (positional, avoids dup-ticker issues)
data_raw = raw.iloc[5:].copy()
data_raw.columns = ['date'] + list(range(len(meta)))
data_raw['date'] = pd.to_datetime(data_raw['date'], dayfirst=True, errors='coerce')
data_raw = data_raw.dropna(subset=['date']).set_index('date')
data_raw = data_raw[~data_raw.index.duplicated(keep='first')]  # drop dup dates
data_raw = data_raw.apply(pd.to_numeric, errors='coerce').ffill(limit=5)

meta

In [ ]:
cmt_dfs = {}
for ctype in meta['type'].unique():
    grp = meta[meta['type'] == ctype].copy()          # subset of meta
    panel = data_raw[grp.index.tolist()].dropna(how='all')  # positional columns
    df = build_country_cmt(panel.index.tolist(), panel, grp)
    cmt_dfs[f'df_{ctype.lower()}_cmt'] = df

df_perugb_cmt = cmt_dfs['df_perugb_cmt']
df_coltes_cmt = cmt_dfs['df_coltes_cmt']
df_mbono_cmt  = cmt_dfs['df_mbono_cmt']
df_bntnf_cmt  = cmt_dfs['df_bntnf_cmt']
df_btpcl_cmt  = cmt_dfs['df_btpcl_cmt']

for k, v in cmt_dfs.items():
    print(k, v.shape)
df_perugb_cmt.head()

In [ ]:
dc_map = meta.drop_duplicates('type').set_index('type')['daycount'].to_dict()


def convert_360(df, dc):
    # in-place conversion to 360d annual effective; dc == 360 is a no-op
    if dc == 365:
        df[TENOR_COLS] = ((1 + df[TENOR_COLS] / 100) ** (365 / 360) - 1) * 100
    elif dc == 252:
        df[TENOR_COLS] = ((1 + df[TENOR_COLS] / 100) ** (252 / 360) - 1) * 100


for ctype in meta['type'].unique():
    convert_360(cmt_dfs[f'df_{ctype.lower()}_cmt'], dc_map[ctype])

# re-assign named variables after in-place conversion
df_perugb_cmt = cmt_dfs['df_perugb_cmt']
df_coltes_cmt = cmt_dfs['df_coltes_cmt']
df_mbono_cmt  = cmt_dfs['df_mbono_cmt']
df_bntnf_cmt  = cmt_dfs['df_bntnf_cmt']
df_btpcl_cmt  = cmt_dfs['df_btpcl_cmt']

df_perugb_cmt.head()

In [ ]:
for k, v in cmt_dfs.items():
    print(f'{k}: {v.shape}')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# set df_plot to whichever CMT df you want to graph
df_plot = df_perugb_cmt  # <- change here

fig, ax = plt.subplots(figsize=(14, 5))
for c in TENOR_COLS:
    ax.plot(df_plot['Fecha'], df_plot[c], linewidth=0.9, label=c)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate(rotation=45)
ax.set_ylabel('Yield (%)')
ax.set_title(df_plot.attrs.get('label', 'CMT Curve — all tenors'))
ax.legend(loc='upper left', ncol=4, fontsize=8, framealpha=0.5)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()